# Algorithms from Scratch

**Course:** [ML in Practice](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/03-algorithms-from-scratch)

This notebook implements core ML algorithms from scratch in NumPy: linear regression (normal equation + gradient descent), logistic regression, K-Means clustering, and decision tree splits.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Linear Regression: Normal Equation vs Gradient Descent

In [ ]:
class LinearRegressionNE:
    """Linear regression via normal equation: θ = (X'X)^{-1} X'y."""
    
    def fit(self, X, y, ridge_lambda=1e-6):
        # Add bias column
        Xb = np.column_stack([np.ones(len(X)), X])
        # Ridge: (X'X + λI)^{-1} X'y for numerical stability
        I = np.eye(Xb.shape[1])
        I[0, 0] = 0  # don't regularize bias
        self.theta = np.linalg.solve(Xb.T @ Xb + ridge_lambda * I, Xb.T @ y)
        return self
    
    def predict(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return Xb @ self.theta

class LinearRegressionGD:
    """Linear regression via gradient descent."""
    
    def fit(self, X, y, lr=0.01, n_epochs=1000):
        Xb = np.column_stack([np.ones(len(X)), X])
        self.theta = np.zeros(Xb.shape[1])
        m = len(y)
        self.losses = []
        for _ in range(n_epochs):
            grad = Xb.T @ (Xb @ self.theta - y) / m
            self.theta -= lr * grad
            self.losses.append(np.mean((Xb @ self.theta - y)**2))
        return self
    
    def predict(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return Xb @ self.theta

# Generate data
X = np.random.randn(100, 2)
y = 3 * X[:, 0] - 2 * X[:, 1] + 1 + np.random.randn(100) * 0.5

ne = LinearRegressionNE().fit(X, y)
gd = LinearRegressionGD().fit(X, y)

print("True coefficients: [bias=1, x1=3, x2=-2]")
print(f"Normal equation:   {ne.theta.round(3)}")
print(f"Gradient descent:  {gd.theta.round(3)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(gd.losses, color='#6366f1', linewidth=2)
ax.axhline(np.mean((ne.predict(X) - y)**2), color='#2dd4bf', linestyle='--',
           label=f'Normal equation MSE (optimal)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Gradient descent converges to normal equation solution', fontsize=11)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
plt.tight_layout()
plt.show()

## K-Means from scratch

In [ ]:
class KMeans:
    def __init__(self, k=3, max_iter=100, tol=1e-4):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
    
    def fit(self, X):
        # K-means++ initialization
        centroids = [X[np.random.randint(len(X))]]
        for _ in range(self.k - 1):
            dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
            probs = dists / dists.sum()
            centroids.append(X[np.random.choice(len(X), p=probs)])
        self.centroids = np.array(centroids)
        
        self.inertias = []
        for _ in range(self.max_iter):
            # Assign
            dists = np.array([np.sum((X - c)**2, axis=1) for c in self.centroids])
            labels = np.argmin(dists, axis=0)
            # Update
            new_centroids = np.array([X[labels == k].mean(0) if (labels == k).any()
                                      else self.centroids[k] for k in range(self.k)])
            inertia = sum(np.sum((X[labels == k] - new_centroids[k])**2)
                         for k in range(self.k) if (labels == k).any())
            self.inertias.append(inertia)
            if np.max(np.abs(new_centroids - self.centroids)) < self.tol:
                break
            self.centroids = new_centroids
        self.labels_ = labels
        return self

# Generate 3-cluster data
centers = [[-3, -3], [0, 3], [3, -2]]
X_km = np.vstack([np.random.randn(100, 2) + c for c in centers])
km = KMeans(k=3).fit(X_km)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#6366f1', '#2dd4bf', '#f97316']

axes[0].scatter(X_km[:, 0], X_km[:, 1], c=[colors[l] for l in km.labels_], alpha=0.6, s=20)
axes[0].scatter(km.centroids[:, 0], km.centroids[:, 1], c='white', s=150, zorder=5, marker='*')
axes[0].set_title('K-Means clustering (K=3)', fontsize=11)

axes[1].plot(km.inertias, 'o-', color='#6366f1', linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Inertia (within-cluster variance)')
axes[1].set_title('K-Means convergence', fontsize=11)

plt.tight_layout()
plt.show()

## Decision tree: Gini impurity and best split

In [ ]:
def gini(y):
    """Gini impurity = 1 - sum(p_k^2)."""
    if len(y) == 0:
        return 0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs**2)

def information_gain(y, y_left, y_right):
    n = len(y)
    return gini(y) - (len(y_left)/n * gini(y_left) + len(y_right)/n * gini(y_right))

def best_split(X_col, y):
    """Find best threshold for a single feature column."""
    best_gain, best_thresh = -1, None
    thresholds = np.unique(X_col)
    for t in thresholds[:-1]:
        left = y[X_col <= t]
        right = y[X_col > t]
        gain = information_gain(y, left, right)
        if gain > best_gain:
            best_gain = gain
            best_thresh = t
    return best_thresh, best_gain

# Demo
X_tree = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y_tree = np.array([0, 0, 0, 0, 1, 1, 1, 1])

thresh, gain = best_split(X_tree, y_tree)
print(f"Best split threshold: {thresh}, information gain: {gain:.4f}")
print(f"Root Gini: {gini(y_tree):.4f}")
print(f"After split: left Gini={gini(y_tree[X_tree<=thresh]):.4f}, right Gini={gini(y_tree[X_tree>thresh]):.4f}")

## ✏️ Your turn

### Exercise 1: Implement ridge regression

Add L2 regularization to the normal equation: θ = (XᵀX + λI)⁻¹Xᵀy.

In [ ]:
def ridge_regression(X, y, lambda_reg):
    """
    Fit ridge regression using the regularized normal equation.
    
    Args:
        X: np.ndarray (n, p) — feature matrix (WITHOUT bias column)
        y: np.ndarray (n,) — target
        lambda_reg: float — L2 regularization strength
    Returns:
        np.ndarray: weight vector of shape (p+1,) — [bias, w1, w2, ...]
    """
    # TODO(you): add bias column, form the regularized normal equation
    # Don't regularize the bias term (identity row/col for features only)
    pass


# Test
theta_no_reg = ridge_regression(X, y, lambda_reg=0)
theta_reg = ridge_regression(X, y, lambda_reg=10)
print(f"No regularization:  {theta_no_reg.round(3)}")
print(f"With regularization: {theta_reg.round(3)}")
print(f"True values:         [1.0, 3.0, -2.0]")

In [ ]:
theta_no_reg = ridge_regression(X, y, 0)
theta_reg = ridge_regression(X, y, 100)
assert theta_no_reg is not None, "Should return weights"
assert theta_no_reg.shape == (3,), f"Expected shape (3,), got {theta_no_reg.shape}"
# High regularization should shrink feature weights toward 0
assert np.abs(theta_reg[1:]).max() < np.abs(theta_no_reg[1:]).max(), \
    "Stronger regularization should shrink feature weights"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ridge_regression(X, y, lambda_reg):
    Xb = np.column_stack([np.ones(len(X)), X])
    p = Xb.shape[1]
    I = np.eye(p)
    I[0, 0] = 0  # don't regularize bias
    return np.linalg.solve(Xb.T @ Xb + lambda_reg * I, Xb.T @ y)
```
</details>

### Exercise 2: Implement Gini impurity for a split

Given a feature column and labels, compute the weighted Gini impurity after splitting at a given threshold.

In [ ]:
def gini_after_split(feature_col, labels, threshold):
    """
    Compute the weighted Gini impurity of a split.
    
    Weighted Gini = (n_left/n) * Gini(left) + (n_right/n) * Gini(right)
    
    Args:
        feature_col: np.ndarray (n,) — feature values
        labels: np.ndarray (n,) — class labels
        threshold: float — split point (left: ≤ threshold, right: > threshold)
    Returns:
        float: weighted Gini impurity, lower is better
    """
    # TODO(you): split labels into left and right using threshold
    # return weighted average of Gini impurities
    pass


feat = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
labs = np.array([0, 0, 0, 1, 1, 1])
print(f"Split at 3: {gini_after_split(feat, labs, 3):.4f} (should be near 0 — perfect split)")
print(f"Split at 1: {gini_after_split(feat, labs, 1):.4f} (poor split — all remaining are mixed)")

In [ ]:
feat = np.array([1., 2., 3., 4., 5., 6.])
labs = np.array([0, 0, 0, 1, 1, 1])
g_perfect = gini_after_split(feat, labs, 3)
g_poor = gini_after_split(feat, labs, 1)
assert g_perfect is not None, "Should return a float"
assert g_perfect < g_poor, "Perfect split should have lower weighted Gini than poor split"
assert abs(g_perfect) < 0.01, f"Perfect split (3/3 homogeneous groups) should give ~0 Gini, got {g_perfect}"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gini_after_split(feature_col, labels, threshold):
    n = len(labels)
    left_mask = feature_col <= threshold
    right_mask = ~left_mask
    return (left_mask.sum() / n) * gini(labels[left_mask]) + \
           (right_mask.sum() / n) * gini(labels[right_mask])
```
</details>